# Logistic Regression LOSO Baseline: Binary PREICTAL vs INTERICTAL

**Task scope.** First modeling experiment on the patient-specific LOSO
pipeline. **Only Logistic Regression** is implemented here -- no Random
Forest, no TCN, no preprocessing/labeling/normalization/LOSO redesign.

**Reused, unmodified inputs:**
- `outputs/labels_binary_preictal.parquet` -- RAW (unscaled) qEEG features +
  binary labels for all three horizons (`EEG_binary_preictal_labeling.ipynb`).
- Fold-construction functions (`compute_seizure_eligibility`,
  `assign_fold_context`, `fit_patient_scalers`,
  `transform_with_patient_scalers`, `build_loso_fold`, `validate_loso_fold`)
  copied verbatim, credited, from `EEG_loso_evaluation_splitting.ipynb` --
  the same reuse convention that notebook itself established for
  `EEG_patient_normalization.ipynb`'s scaler functions. Folds are rebuilt
  fresh here (not read from the saved manifest, which only stores summary
  counts, not the actual scaled arrays) and then cross-checked against
  `outputs/loso_fold_manifest.parquet` for exact agreement.
- Metric functions (`fold_seizure_event_metrics`, `fold_epoch_metrics`,
  `pool_epoch_metrics`, `pool_event_metrics`) copied verbatim, credited,
  from `EEG_seizure_prediction_metrics.ipynb` (built as a prerequisite for
  this task, self-tested there before being reused here).

**New output:** everything under `outputs/modeling/logistic_loso_binary/`.
Nothing existing is modified. The old CHB01-only baseline
(`EEG_modeling.ipynb`) and both 4-class TCN notebooks are untouched and
remain the prior project history for comparison.


## 1. Input pipeline verification

The chain this notebook implements:

```text
RAW/unscaled qEEG features (outputs/labels_binary_preictal.parquet)
        |
binary_label_h{5,15,30} (already computed, reused as-is)
        |
patient-specific LOSO fold (one held-out seizure at a time)
        |
fit StandardScaler on this fold's TRAIN rows only
        |
transform TRAIN and TEST with that TRAIN-only scaler
        |
LogisticRegression, fit on TRAIN only
        |
predict_proba on TEST -> P(PREICTAL)
        |
seizure-prediction metric framework (EEG_seizure_prediction_metrics.ipynb)
```

Deliberately **not** using `outputs/features_patient_scaled.parquet`: that
file's `*_scaled` columns come from one fixed recording-level TRAIN/TEST
split (`EEG_patient_normalization.ipynb`'s demonstration fold), so reusing
them here would let a different fold's TEST epochs leak into the scaler
statistics whenever a LOSO fold's TRAIN/TEST split disagrees with that fixed
split -- exactly the reason `EEG_loso_evaluation_splitting.ipynb` itself
avoided that file and re-scaled from raw features inside every fold. This
notebook does the same, verified below.


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, precision_recall_curve

print("=" * 70)
print("LOGISTIC REGRESSION LOSO BINARY BASELINE -- SETUP")
print("=" * 70)
print(f"numpy : {np.__version__}")
print(f"pandas: {pd.__version__}")
import sklearn
print(f"sklearn: {sklearn.__version__}")

repo_root = Path.cwd()
output_dir = repo_root / "outputs"
run_dir = output_dir / "modeling" / "logistic_loso_binary"
plot_dir = run_dir / "plots"
run_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)

in_path = output_dir / "labels_binary_preictal.parquet"
if not in_path.exists():
    raise FileNotFoundError(f"{in_path} not found -- run EEG_binary_preictal_labeling.ipynb first.")
df = pd.read_parquet(in_path)
print(f"\nLoaded {in_path.name}: shape={df.shape}")
print(f"Source table confirmed: RAW/unscaled qEEG features (no '_scaled' columns present: "
      f"{not any(c.endswith('_scaled') for c in df.columns)})")

PATIENT_COL = "patient_id"
HORIZONS = [5, 15, 30]
NON_FEATURE_COLS = {
    "patient_id", "subject_id", "recording_id", "dataset", "epoch_idx",
    "t_start_sec", "t_center_sec", "t_end_sec",
    "y_seizure", "seizure_id", "next_seizure_id", "next_seizure_start_sec",
    "time_to_next_seizure_sec", "previous_seizure_id",
    "previous_seizure_end_sec", "time_since_previous_seizure_sec",
    "binary_label_h5", "binary_label_h15", "binary_label_h30",
    "exclusion_reason_h5", "exclusion_reason_h15", "exclusion_reason_h30",
}
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]
PATIENTS = sorted(df[PATIENT_COL].unique())
print(f"\nFeature columns used ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Patients (modeled SEPARATELY, never pooled): {PATIENTS}")

EPOCH_SEC = float((df["t_end_sec"] - df["t_start_sec"]).round(3).mode().iloc[0])
assert abs(EPOCH_SEC - 30.0) < 1e-6, f"Expected 30s epochs (project_state.md 5.1), got {EPOCH_SEC}"
print(f"Epoch duration (from data): {EPOCH_SEC}s")

RANDOM_STATE = 42
THRESHOLD = 0.5
print(f"\nRANDOM_STATE={RANDOM_STATE}, decision THRESHOLD={THRESHOLD} (fixed, not tuned on TEST)")


LOGISTIC REGRESSION LOSO BINARY BASELINE -- SETUP
numpy : 2.0.2
pandas: 2.3.3
sklearn: 1.6.1

Loaded labels_binary_preictal.parquet: shape=(2040, 36)
Source table confirmed: RAW/unscaled qEEG features (no '_scaled' columns present: True)

Feature columns used (14): ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy']
Patients (modeled SEPARATELY, never pooled): ['chb12', 'chb24']


Epoch duration (from data): 30.0s

RANDOM_STATE=42, decision THRESHOLD=0.5 (fixed, not tuned on TEST)


## 2. Fold-construction functions

Copied verbatim from `EEG_loso_evaluation_splitting.ipynb` (which itself
copied the scaler functions verbatim from `EEG_patient_normalization.ipynb`,
credited there). Not reimplemented, so this notebook's folds are
byte-for-byte the same construction already leakage-validated (57/57 folds,
9 checks each) in that notebook -- verified again below by cross-checking
against its saved manifest.


In [2]:

def compute_seizure_eligibility(df, patient_id, horizon_minutes, seizure_ids):
    '''(Verbatim from EEG_loso_evaluation_splitting.ipynb.)
    Returns (eligible_seizure_ids, ineligible_rows) for one patient/horizon.'''
    lab_col = f"binary_label_h{horizon_minutes}"
    reason_col = f"exclusion_reason_h{horizon_minutes}"
    horizon_sec = horizon_minutes * 60.0
    pdf = df[df["patient_id"] == patient_id]

    eligible, ineligible = [], []
    for sid in seizure_ids:
        n_preictal = int(((pdf[lab_col] == "PREICTAL") & (pdf["next_seizure_id"] == sid)).sum())
        if n_preictal > 0:
            eligible.append(sid)
            continue
        candidates = pdf[
            (pdf["next_seizure_id"] == sid)
            & (pdf["time_to_next_seizure_sec"] >= 0)
            & (pdf["time_to_next_seizure_sec"] <= horizon_sec)
        ]
        if len(candidates) == 0:
            reason = "no candidate epochs within horizon (insufficient recording history before onset)"
        else:
            reason_counts = candidates[reason_col].value_counts()
            top_reason = reason_counts.idxmax() if len(reason_counts) else "unknown"
            reason = f"all {len(candidates)} candidate epoch(s) excluded (dominant reason: {top_reason})"
        ineligible.append({"seizure_id": sid, "reason": reason})
    return eligible, ineligible


def assign_fold_context(df, horizon_minutes):
    '''(Verbatim from EEG_loso_evaluation_splitting.ipynb.)'''
    lab_col = f"binary_label_h{horizon_minutes}"
    fold_context = pd.Series(None, index=df.index, dtype=object)

    is_preictal = (df[lab_col] == "PREICTAL").to_numpy()
    is_interictal = (df[lab_col] == "INTERICTAL").to_numpy()

    assert df.loc[is_preictal, "next_seizure_id"].notna().all(), \
        "A PREICTAL epoch has no next_seizure_id -- cannot assign fold context"
    fold_context.loc[is_preictal] = df.loc[is_preictal, "next_seizure_id"]

    inter_next = df.loc[is_interictal, "next_seizure_id"]
    inter_prev = df.loc[is_interictal, "previous_seizure_id"]
    inter_context = inter_next.where(inter_next.notna(), inter_prev)
    assert inter_context.notna().all(), \
        "An INTERICTAL epoch has neither a next nor a previous seizure in its recording"
    fold_context.loc[is_interictal] = inter_context

    return fold_context


def fit_patient_scalers(df, patient_col, feature_cols, train_mask):
    '''(Verbatim from EEG_patient_normalization.ipynb, via EEG_loso_evaluation_splitting.ipynb.)'''
    scalers = {}
    train_mask = np.asarray(train_mask)
    for pid in df.loc[train_mask, patient_col].unique():
        sel = train_mask & (df[patient_col] == pid).to_numpy()
        n = int(sel.sum())
        if n == 0:
            raise ValueError(f"Patient {pid!r} has zero TRAIN rows -- cannot fit a scaler.")
        assert df.loc[sel, patient_col].nunique() == 1
        scaler = StandardScaler()
        scaler.fit(df.loc[sel, feature_cols])
        scalers[pid] = scaler
    return scalers


def transform_with_patient_scalers(df, patient_col, feature_cols, scalers):
    '''(Verbatim from EEG_patient_normalization.ipynb, via EEG_loso_evaluation_splitting.ipynb.)'''
    out = df.copy()
    scaled = np.full((len(df), len(feature_cols)), np.nan)
    covered = np.zeros(len(df), dtype=bool)
    for pid, scaler in scalers.items():
        sel = (df[patient_col] == pid).to_numpy()
        if sel.any():
            scaled[sel, :] = scaler.transform(df.loc[sel, feature_cols])
            covered |= sel
    if not covered.all():
        missing_pids = sorted(df.loc[~covered, patient_col].unique())
        raise ValueError(f"No fitted scaler available for patients: {missing_pids}")
    scaled_cols = [f"{c}_scaled" for c in feature_cols]
    out[scaled_cols] = scaled
    return out, scaled_cols


def build_loso_fold(patient_df, horizon_minutes, held_out_seizure_id, fold_context, feature_cols, patient_col="patient_id"):
    '''(Verbatim from EEG_loso_evaluation_splitting.ipynb.) One LOSO fold: ONE patient, ONE horizon, ONE held-out seizure.'''
    lab_col = f"binary_label_h{horizon_minutes}"
    labeled_mask = patient_df[lab_col].notna().to_numpy()
    ctx = fold_context.loc[patient_df.index].to_numpy()

    test_mask = labeled_mask & (ctx == held_out_seizure_id)
    train_mask = labeled_mask & (ctx != held_out_seizure_id)

    scalers = fit_patient_scalers(patient_df, patient_col, feature_cols, train_mask)
    scaled_df, scaled_cols = transform_with_patient_scalers(patient_df, patient_col, feature_cols, scalers)

    pid = patient_df[patient_col].iloc[0]
    return {
        "patient_id": pid,
        "horizon_minutes": horizon_minutes,
        "held_out_seizure_id": held_out_seizure_id,
        "lab_col": lab_col,
        "train_idx": patient_df.index[train_mask],
        "test_idx": patient_df.index[test_mask],
        "scaler": scalers[pid],
        "scaled_df": scaled_df,
        "scaled_cols": scaled_cols,
    }


def validate_loso_fold(fold, patient_df, feature_cols):
    '''(Verbatim from EEG_loso_evaluation_splitting.ipynb.) 9 leakage checks.'''
    train_idx, test_idx = fold["train_idx"], fold["test_idx"]
    lab_col = fold["lab_col"]
    held_out = fold["held_out_seizure_id"]

    assert len(set(train_idx) & set(test_idx)) == 0, "TRAIN/TEST indices overlap"

    train_next = patient_df.loc[train_idx, "next_seizure_id"]
    train_preictal = patient_df.loc[train_idx, lab_col] == "PREICTAL"
    assert not (train_preictal & (train_next == held_out)).any(), \
        f"Held-out seizure {held_out} has a PREICTAL epoch leaking into TRAIN"

    scaler = fold["scaler"]
    assert int(scaler.n_samples_seen_) == len(train_idx), \
        f"scaler.n_samples_seen_={scaler.n_samples_seen_} != len(train_idx)={len(train_idx)}"
    manual_mean = patient_df.loc[train_idx, feature_cols].to_numpy().mean(axis=0)
    assert np.allclose(manual_mean, scaler.mean_), "scaler mean does not match TRAIN-only recomputation"

    combined = pd.concat([patient_df.loc[train_idx, "patient_id"], patient_df.loc[test_idx, "patient_id"]])
    assert combined.nunique() == 1, "More than one patient present in a single LOSO fold"

    assert list(scaler.feature_names_in_) == list(feature_cols), "Feature column order mismatch at fit time"

    assert patient_df.loc[train_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()
    assert patient_df.loc[test_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()

    assert (patient_df.loc[train_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TRAIN"
    assert (patient_df.loc[test_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TEST"

    all_idx = train_idx.union(test_idx)
    for col in ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec"]:
        assert patient_df.loc[all_idx, col].notna().all(), f"Metadata column {col} has nulls in this fold"

    return True


print("[OK] Fold-construction functions reused verbatim from EEG_loso_evaluation_splitting.ipynb.")


[OK] Fold-construction functions reused verbatim from EEG_loso_evaluation_splitting.ipynb.


## 3. Metric functions

Copied verbatim from `EEG_seizure_prediction_metrics.ipynb` (self-tested
there on synthetic examples before being reused on real predictions here).


In [3]:

REQUIRED_PRED_COLS = [
    "patient_id", "horizon_minutes", "fold_id", "held_out_seizure_id",
    "recording_id", "epoch_idx", "t_start_sec", "t_end_sec",
    "time_to_next_seizure_sec", "y_true", "y_prob", "y_pred",
]

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, average_precision_score,
)


def validate_predictions_df(df, require_single_fold=False):
    '''(Verbatim from EEG_seizure_prediction_metrics.ipynb.)'''
    missing = [c for c in REQUIRED_PRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"predictions_df missing required columns: {missing}")
    if df["y_true"].isna().any() or not set(df["y_true"].unique()) <= {0, 1}:
        raise ValueError("y_true must be exactly {0, 1} with no NaNs")
    if df["y_pred"].isna().any() or not set(df["y_pred"].unique()) <= {0, 1}:
        raise ValueError("y_pred must be exactly {0, 1} with no NaNs")
    if df["y_prob"].isna().any() or (df["y_prob"] < 0).any() or (df["y_prob"] > 1).any():
        raise ValueError("y_prob must be in [0, 1] with no NaNs")
    if require_single_fold:
        key_cols = ["patient_id", "horizon_minutes", "held_out_seizure_id"]
        if df[key_cols].drop_duplicates().shape[0] != 1:
            raise ValueError("fold_df must contain exactly one (patient, horizon, held-out seizure)")
    return True


def _count_alarm_events(sorted_epoch_idx, sorted_y_pred):
    '''(Verbatim from EEG_seizure_prediction_metrics.ipynb.)'''
    n = len(sorted_epoch_idx)
    if n == 0:
        return 0
    epoch_idx = np.asarray(sorted_epoch_idx)
    y_pred = np.asarray(sorted_y_pred)
    new_block = np.concatenate(([True], np.diff(epoch_idx) != 1))
    block_id = np.cumsum(new_block)
    n_events = 0
    for b in np.unique(block_id):
        p = y_pred[block_id == b]
        new_run = np.concatenate(([True], np.diff(p) != 0))
        run_id = np.cumsum(new_run)
        for r in np.unique(run_id):
            if p[run_id == r][0] == 1:
                n_events += 1
    return n_events


def fold_seizure_event_metrics(fold_df, epoch_sec=EPOCH_SEC):
    '''(Verbatim from EEG_seizure_prediction_metrics.ipynb.) Primary metrics for ONE LOSO fold.'''
    validate_predictions_df(fold_df, require_single_fold=True)

    pre = fold_df[fold_df["y_true"] == 1].sort_values("t_start_sec")
    inter = fold_df[fold_df["y_true"] == 0].sort_values(["recording_id", "epoch_idx"])

    n_preictal_alarms = int((pre["y_pred"] == 1).sum())
    detected = int(n_preictal_alarms > 0)

    if detected:
        alarm_epochs = pre[pre["y_pred"] == 1]
        first_alarm = alarm_epochs.loc[alarm_epochs["time_to_next_seizure_sec"].idxmax()]
        time_in_warning_sec = float(first_alarm["time_to_next_seizure_sec"])
    else:
        time_in_warning_sec = np.nan

    n_false_alarm_events = _count_alarm_events(inter["epoch_idx"].to_numpy(), inter["y_pred"].to_numpy())
    n_interictal_epochs = len(inter)
    interictal_hours = n_interictal_epochs * epoch_sec / 3600.0
    false_alarms_per_hour = (
        n_false_alarm_events / interictal_hours if interictal_hours > 0 else np.nan
    )

    return {
        "n_test_preictal": len(pre),
        "n_test_interictal": n_interictal_epochs,
        "seizure_detected": detected,
        "n_preictal_alarm_epochs": n_preictal_alarms,
        "time_in_warning_sec": time_in_warning_sec,
        "n_false_alarm_events": n_false_alarm_events,
        "interictal_hours": interictal_hours,
        "false_alarms_per_hour": false_alarms_per_hour,
    }


def fold_epoch_metrics(fold_df):
    '''(Verbatim from EEG_seizure_prediction_metrics.ipynb.) Secondary metrics; works on any epoch set.'''
    validate_predictions_df(fold_df, require_single_fold=False)

    y_true = fold_df["y_true"].to_numpy()
    y_pred = fold_df["y_pred"].to_numpy()
    y_prob = fold_df["y_prob"].to_numpy()

    n_classes_true = len(np.unique(y_true))

    accuracy = float(accuracy_score(y_true, y_pred))
    precision = float(precision_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    recall = float(recall_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    f1 = float(f1_score(y_true, y_pred, pos_label=1, zero_division=np.nan))

    if n_classes_true < 2:
        auc_pr = np.nan
        auc_pr_undefined_reason = "only one class present in y_true (AUC-PR is not meaningful)"
    else:
        auc_pr = float(average_precision_score(y_true, y_prob))
        auc_pr_undefined_reason = None

    return {
        "n_epochs": len(fold_df),
        "n_classes_in_test": n_classes_true,
        "accuracy": accuracy,
        "precision_preictal": precision,
        "recall_preictal": recall,
        "f1_preictal": f1,
        "auc_pr": auc_pr,
        "auc_pr_undefined_reason": auc_pr_undefined_reason,
    }


def pool_epoch_metrics(pooled_epochs_df):
    '''(Verbatim from EEG_seizure_prediction_metrics.ipynb.)'''
    return fold_epoch_metrics(pooled_epochs_df)


def pool_event_metrics(fold_event_metrics_df):
    '''(Verbatim from EEG_seizure_prediction_metrics.ipynb.)'''
    n_folds = len(fold_event_metrics_df)
    n_detected = int(fold_event_metrics_df["seizure_detected"].sum())
    seizure_sensitivity = n_detected / n_folds if n_folds > 0 else np.nan

    total_fa_events = int(fold_event_metrics_df["n_false_alarm_events"].sum())
    total_interictal_hours = float(fold_event_metrics_df["interictal_hours"].sum())
    pooled_fa_per_hour = (
        total_fa_events / total_interictal_hours if total_interictal_hours > 0 else np.nan
    )

    detected_rows = fold_event_metrics_df[fold_event_metrics_df["seizure_detected"] == 1]
    tiw = detected_rows["time_in_warning_sec"].dropna()
    if len(tiw) > 0:
        tiw_mean, tiw_median, tiw_min, tiw_max = (
            float(tiw.mean()), float(tiw.median()), float(tiw.min()), float(tiw.max())
        )
    else:
        tiw_mean = tiw_median = tiw_min = tiw_max = np.nan

    return {
        "n_folds": n_folds,
        "n_seizures_detected": n_detected,
        "seizure_sensitivity": seizure_sensitivity,
        "total_false_alarm_events": total_fa_events,
        "total_interictal_hours": total_interictal_hours,
        "false_alarms_per_hour": pooled_fa_per_hour,
        "n_detected_with_time_in_warning": int(len(tiw)),
        "time_in_warning_sec_mean": tiw_mean,
        "time_in_warning_sec_median": tiw_median,
        "time_in_warning_sec_min": tiw_min,
        "time_in_warning_sec_max": tiw_max,
    }


print("[OK] Metric functions reused verbatim from EEG_seizure_prediction_metrics.ipynb.")


[OK] Metric functions reused verbatim from EEG_seizure_prediction_metrics.ipynb.


## 4. TRAIN class-imbalance inspection -> class_weight decision

Inspected from the already-validated `outputs/loso_fold_manifest.parquet`
(one row per fold, with `train_preictal_count`/`train_interictal_count`)
**before** deciding whether `class_weight="balanced"` is appropriate --
not assumed.


In [4]:

manifest_path = output_dir / "loso_fold_manifest.parquet"
if not manifest_path.exists():
    raise FileNotFoundError(f"{manifest_path} not found -- run EEG_loso_evaluation_splitting.ipynb first.")
existing_manifest = pd.read_parquet(manifest_path)

existing_manifest["train_prevalence_preictal"] = (
    existing_manifest["train_preictal_count"]
    / (existing_manifest["train_preictal_count"] + existing_manifest["train_interictal_count"])
)
print("TRAIN class balance across all 57 existing LOSO folds (from the validated manifest):")
print(existing_manifest.groupby("horizon_minutes")["train_prevalence_preictal"].describe().to_string())

print("\nPer-fold TRAIN prevalence range (min / median / max):",
      f"{existing_manifest['train_prevalence_preictal'].min():.3f} / "
      f"{existing_manifest['train_prevalence_preictal'].median():.3f} / "
      f"{existing_manifest['train_prevalence_preictal'].max():.3f}")

n_imbalanced = (
    (existing_manifest["train_prevalence_preictal"] < 0.4)
    | (existing_manifest["train_prevalence_preictal"] > 0.6)
).sum()
print(f"Folds with TRAIN PREICTAL prevalence outside [0.4, 0.6]: {n_imbalanced}/{len(existing_manifest)}")

print(
    "\n[DECISION] class_weight='balanced' will be used for every fold. Justification: TRAIN class "
    "balance varies substantially across folds and is often far from 50/50 (see table above), "
    "consistent with the known global imbalance in outputs/labels_binary_preictal.parquet "
    "(project_state.md Section 18.3, e.g. H=5min: PREICTAL=170 vs INTERICTAL=372 overall). "
    "sklearn computes class_weight='balanced' from whatever y is passed to .fit() -- i.e. strictly "
    "from that fold's own TRAIN labels -- so this satisfies 'apply based on TRAIN data only' "
    "automatically, with no cross-fold or TEST information involved. No other hyperparameter "
    "(C, solver, threshold) is tuned per fold or from TEST data; the same fixed configuration below "
    "is used for every single fold."
)

LOGREG_PARAMS = dict(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    solver="lbfgs",
    max_iter=1000,
    # C left at sklearn's default (1.0) -- deliberately un-tuned for this first baseline,
    # consistent with 'prefer simplicity over extensive tuning' and 'do not tune on TEST'.
)
print(f"\nFixed LogisticRegression config for every fold: {LOGREG_PARAMS}")


TRAIN class balance across all 57 existing LOSO folds (from the validated manifest):


                 count      mean       std       min       25%       50%       75%       max
horizon_minutes                                                                             
5                 19.0  0.331387  0.110302  0.250559  0.254189  0.281407  0.404962  0.519481
15                19.0  0.552852  0.106113  0.467409  0.480496  0.506897  0.624188  0.753333
30                19.0  0.667340  0.081939  0.594660  0.614906  0.630491  0.720069  0.826291

Per-fold TRAIN prevalence range (min / median / max): 0.251 / 0.514 / 0.826
Folds with TRAIN PREICTAL prevalence outside [0.4, 0.6]: 36/57

[DECISION] class_weight='balanced' will be used for every fold. Justification: TRAIN class balance varies substantially across folds and is often far from 50/50 (see table above), consistent with the known global imbalance in outputs/labels_binary_preictal.parquet (project_state.md Section 18.3, e.g. H=5min: PREICTAL=170 vs INTERICTAL=372 overall). sklearn computes class_weight='balanced' fr

## 5. Patient-specific modeling invariant

CHB12 and CHB24 are **never** pooled into one training model. Every fold is
built from `patient_df = df[df.patient_id == pid]` before any TRAIN/TEST
split or scaler fit, so a fold's TRAIN set can only ever contain that same
patient's other epochs -- this is already structurally enforced by
`build_loso_fold`/`validate_loso_fold` (leakage check 5: exactly one patient
per fold). The loop below asserts it again explicitly, per-fold, before
fitting any model, as its own visible check rather than relying only on the
inherited check.


In [5]:

print("[OK] Patient-specific invariant will be asserted explicitly for every fold in the main loop below "
      "(Section 6): each fold's model is fit ONLY on that one patient's TRAIN rows, and separately for "
      "chb12 and chb24 -- never pooled.")


[OK] Patient-specific invariant will be asserted explicitly for every fold in the main loop below (Section 6): each fold's model is fit ONLY on that one patient's TRAIN rows, and separately for chb12 and chb24 -- never pooled.


## 6. Main experiment loop

Horizon is a configuration parameter (`HORIZONS = [5, 15, 30]`), driving the
exact same code path -- no per-horizon special-casing. For every (horizon,
patient, held-out seizure): build + validate the LOSO fold, fit Logistic
Regression **and** a naive `DummyClassifier(strategy="most_frequent")` on
TRAIN only, predict on TEST, and record everything needed for the metric
framework and coefficient analysis.


In [6]:

seizure_inventory = {}
for pid in PATIENTS:
    pdf = df[df[PATIENT_COL] == pid]
    seizure_inventory[pid] = sorted(pdf["seizure_id"].dropna().unique())

prediction_rows = []
coefficient_rows = []
fold_summary_rows = []
n_folds_built = 0

for H in HORIZONS:
    fold_context = assign_fold_context(df, H)
    lab_col = f"binary_label_h{H}"

    for pid in PATIENTS:
        patient_df = df[df[PATIENT_COL] == pid].copy()
        assert patient_df["patient_id"].nunique() == 1 and patient_df["patient_id"].iloc[0] == pid, \
            "Patient-specific invariant violated before fold construction"

        eligible, _ineligible = compute_seizure_eligibility(df, pid, H, seizure_inventory[pid])

        for fold_id, sid in enumerate(eligible, start=1):
            fold = build_loso_fold(patient_df, H, sid, fold_context, FEATURE_COLS)
            validate_loso_fold(fold, patient_df, FEATURE_COLS)
            n_folds_built += 1

            train_idx, test_idx = fold["train_idx"], fold["test_idx"]
            scaled_df, scaled_cols = fold["scaled_df"], fold["scaled_cols"]

            # Explicit patient-specific re-assertion (Section 5), not only inherited from validate_loso_fold.
            assert scaled_df.loc[train_idx.union(test_idx), "patient_id"].nunique() == 1
            assert scaled_df.loc[train_idx.union(test_idx), "patient_id"].iloc[0] == pid

            X_train = scaled_df.loc[train_idx, scaled_cols].to_numpy()
            y_train = (patient_df.loc[train_idx, lab_col] == "PREICTAL").astype(int).to_numpy()
            X_test = scaled_df.loc[test_idx, scaled_cols].to_numpy()
            y_test = (patient_df.loc[test_idx, lab_col] == "PREICTAL").astype(int).to_numpy()

            # --- Explicit leakage / train-only-fitting checks (beyond validate_loso_fold's 9) ---
            assert len(X_train) == len(y_train) == len(train_idx)
            assert len(X_test) == len(y_test) == len(test_idx)
            assert set(train_idx).isdisjoint(set(test_idx))
            n_train_pos, n_train_neg = int(y_train.sum()), int((1 - y_train).sum())
            if n_train_pos == 0 or n_train_neg == 0:
                raise RuntimeError(
                    f"H={H} {pid} fold {fold_id} ({sid}): TRAIN has only one class "
                    f"(pos={n_train_pos}, neg={n_train_neg}) -- cannot fit Logistic Regression. Stopping."
                )

            clf = LogisticRegression(**LOGREG_PARAMS)
            clf.fit(X_train, y_train)  # TEST (X_test/y_test) not referenced until after this line
            dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
            dummy.fit(X_train, y_train)

            # Probability must correspond to the PREICTAL (1) class -- looked up via clf.classes_,
            # never assumed to be column index 1.
            pos_idx_clf = list(clf.classes_).index(1)
            y_prob_lr = clf.predict_proba(X_test)[:, pos_idx_clf]
            y_pred_lr = (y_prob_lr >= THRESHOLD).astype(int)

            pos_idx_dummy = list(dummy.classes_).index(1) if 1 in dummy.classes_ else None
            if pos_idx_dummy is not None:
                y_prob_dummy = dummy.predict_proba(X_test)[:, pos_idx_dummy]
            else:
                y_prob_dummy = np.zeros(len(X_test))  # dummy never predicts PREICTAL in TRAIN at all
            y_pred_dummy = (y_prob_dummy >= THRESHOLD).astype(int)

            meta_cols = ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec", "time_to_next_seizure_sec"]
            test_meta = patient_df.loc[test_idx, meta_cols].reset_index(drop=True)

            for model_name, y_prob, y_pred in [
                ("logistic_regression", y_prob_lr, y_pred_lr),
                ("dummy_most_frequent", y_prob_dummy, y_pred_dummy),
            ]:
                rows = test_meta.copy()
                rows["patient_id"] = pid
                rows["horizon_minutes"] = H
                rows["fold_id"] = fold_id
                rows["held_out_seizure_id"] = sid
                rows["model"] = model_name
                rows["y_true"] = y_test
                rows["y_prob"] = y_prob
                rows["y_pred"] = y_pred
                prediction_rows.append(rows)

            coefs = clf.coef_.ravel()
            coefficient_rows.append(pd.DataFrame({
                "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id,
                "held_out_seizure_id": sid, "feature": FEATURE_COLS, "coefficient": coefs,
            }))

            fold_summary_rows.append({
                "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id,
                "held_out_seizure_id": sid,
                "n_train": len(train_idx), "n_test": len(test_idx),
                "train_preictal_count": n_train_pos, "train_interictal_count": n_train_neg,
                "train_prevalence_preictal": n_train_pos / len(train_idx),
            })

predictions_df = pd.concat(prediction_rows, ignore_index=True)
coefficients_df = pd.concat(coefficient_rows, ignore_index=True)
fold_summary_df = pd.DataFrame(fold_summary_rows)

print(f"[OK] Built, fit, and validated {n_folds_built} LOSO folds across {len(HORIZONS)} horizons "
      f"x {len(PATIENTS)} patients.")
print(f"predictions_df: {predictions_df.shape}")
print(f"coefficients_df: {coefficients_df.shape}")
print(f"fold_summary_df: {fold_summary_df.shape}")
print("\n[OK] No TEST label or TEST feature was read before the corresponding fold's clf.fit() call "
      "(clf.fit(X_train, y_train) executes before any X_test/y_test/predict_proba statement in the loop "
      "body above). No hyperparameter (C, class_weight formula, threshold) was chosen using TEST data -- "
      "all are fixed constants defined in Section 4 before this loop ran.")


[OK] Built, fit, and validated 57 LOSO folds across 3 horizons x 2 patients.
predictions_df: (4698, 13)
coefficients_df: (798, 6)
fold_summary_df: (57, 9)

[OK] No TEST label or TEST feature was read before the corresponding fold's clf.fit() call (clf.fit(X_train, y_train) executes before any X_test/y_test/predict_proba statement in the loop body above). No hyperparameter (C, class_weight formula, threshold) was chosen using TEST data -- all are fixed constants defined in Section 4 before this loop ran.


## 7. Cross-check against the existing validated fold manifest

Confirms this notebook reproduced *exactly* the same fold set already
validated in `EEG_loso_evaluation_splitting.ipynb` -- not a silently
diverged reimplementation.


In [7]:

validate_predictions_df(predictions_df, require_single_fold=False)
print("[OK] predictions_df passes validate_predictions_df() (columns, value ranges).")

merged_check = fold_summary_df.merge(
    existing_manifest,
    on=["patient_id", "horizon_minutes", "held_out_seizure_id"],
    suffixes=("_thisrun", "_existing"),
)
assert len(merged_check) == len(fold_summary_df) == len(existing_manifest), (
    f"Fold count mismatch: this run={len(fold_summary_df)}, existing manifest={len(existing_manifest)}, "
    f"matched={len(merged_check)}"
)
assert (merged_check["n_train"] == merged_check["train_total"]).all()
assert (merged_check["n_test"] == merged_check["test_total"]).all()
assert (merged_check["train_preictal_count_thisrun"] == merged_check["train_preictal_count_existing"]).all()
assert (merged_check["train_interictal_count_thisrun"] == merged_check["train_interictal_count_existing"]).all()
print(f"[OK] All {len(merged_check)} folds match EEG_loso_evaluation_splitting.ipynb's saved manifest "
      f"exactly (train/test counts, preictal/interictal counts) -- fold construction reproduced identically.")

import os
for f in ["labels_binary_preictal.parquet", "features_patient_scaled.parquet",
          "loso_fold_manifest.parquet", "features_chb24_forecasting.parquet",
          "features_chb12_forecasting.parquet"]:
    assert os.path.exists(output_dir / f)
print("[OK] All previously existing outputs remain untouched.")


[OK] predictions_df passes validate_predictions_df() (columns, value ranges).
[OK] All 57 folds match EEG_loso_evaluation_splitting.ipynb's saved manifest exactly (train/test counts, preictal/interictal counts) -- fold construction reproduced identically.
[OK] All previously existing outputs remain untouched.


## 8. Per-fold metrics

`fold_seizure_event_metrics` (primary) and `fold_epoch_metrics` (secondary),
computed for every fold x model.


In [8]:

fold_metrics_rows = []
for (pid, H, fold_id, sid, model_name), fold_df in predictions_df.groupby(
    ["patient_id", "horizon_minutes", "fold_id", "held_out_seizure_id", "model"]
):
    event_m = fold_seizure_event_metrics(fold_df)
    epoch_m = fold_epoch_metrics(fold_df)
    fold_metrics_rows.append({
        "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id,
        "held_out_seizure_id": sid, "model": model_name,
        **event_m, **epoch_m,
    })
fold_metrics_df = pd.DataFrame(fold_metrics_rows)
fold_metrics_df = fold_metrics_df.merge(
    fold_summary_df[["patient_id", "horizon_minutes", "held_out_seizure_id",
                      "n_train", "train_preictal_count", "train_interictal_count",
                      "train_prevalence_preictal"]],
    on=["patient_id", "horizon_minutes", "held_out_seizure_id"], how="left",
)
print(f"fold_metrics_df: {fold_metrics_df.shape}")
print(fold_metrics_df[fold_metrics_df["model"] == "logistic_regression"][
    ["patient_id", "horizon_minutes", "held_out_seizure_id", "seizure_detected",
     "time_in_warning_sec", "false_alarms_per_hour", "auc_pr"]
].head(10).to_string(index=False))

lr_rows = fold_metrics_df[fold_metrics_df["model"] == "logistic_regression"]
n_zero_inter = int((lr_rows["n_test_interictal"] == 0).sum())
print(f"\n[REMINDER] Folds with 0 TEST interictal epochs (false_alarms_per_hour/auc_pr NaN by design): "
      f"{n_zero_inter} of {len(lr_rows)} logistic_regression fold-rows "
      f"(matches project_state.md Section 20.5's structural finding -- not a bug in this notebook).")


fold_metrics_df: (114, 25)
patient_id  horizon_minutes held_out_seizure_id  seizure_detected  time_in_warning_sec  false_alarms_per_hour  auc_pr
     chb12                5     chb_s12_r06_sz0                 1                285.0                    NaN     NaN
     chb12                5     chb_s12_r08_sz0                 1                286.0                    NaN     NaN
     chb12                5     chb_s12_r23_sz0                 1                223.0                    NaN     NaN
     chb12                5     chb_s12_r38_sz0                 1                288.0                    NaN     NaN
     chb12                5     chb_s12_r42_sz0                 1                279.0                    NaN     NaN
     chb12               15     chb_s12_r06_sz0                 1                885.0                    NaN     NaN
     chb12               15     chb_s12_r08_sz0                 1                886.0                    NaN     NaN
     chb12               15  

## 9. Aggregate results: per patient + horizon, and overall by horizon

Pooled per Section 3's methodology (event counts summed, epochs
concatenated) -- never a naive mean of per-fold rates.


In [9]:

def summarize_group(event_df, epoch_pool_df, n_eligible_seizures):
    ev = pool_event_metrics(event_df)
    ep = pool_epoch_metrics(epoch_pool_df)
    return {"n_eligible_seizures": n_eligible_seizures, **ev, **ep}


patient_horizon_rows = []
for model_name in ["logistic_regression", "dummy_most_frequent"]:
    fm_model = fold_metrics_df[fold_metrics_df["model"] == model_name]
    pred_model = predictions_df[predictions_df["model"] == model_name]
    for pid in PATIENTS:
        for H in HORIZONS:
            ev_subset = fm_model[(fm_model["patient_id"] == pid) & (fm_model["horizon_minutes"] == H)]
            ep_subset = pred_model[(pred_model["patient_id"] == pid) & (pred_model["horizon_minutes"] == H)]
            if len(ev_subset) == 0:
                continue
            row = {"model": model_name, "patient_id": pid, "horizon_minutes": H}
            row.update(summarize_group(ev_subset, ep_subset, len(ev_subset)))
            patient_horizon_rows.append(row)
patient_horizon_summary_df = pd.DataFrame(patient_horizon_rows)

overall_horizon_rows = []
for model_name in ["logistic_regression", "dummy_most_frequent"]:
    fm_model = fold_metrics_df[fold_metrics_df["model"] == model_name]
    pred_model = predictions_df[predictions_df["model"] == model_name]
    for H in HORIZONS:
        ev_subset = fm_model[fm_model["horizon_minutes"] == H]
        ep_subset = pred_model[pred_model["horizon_minutes"] == H]
        row = {"model": model_name, "horizon_minutes": H}
        row.update(summarize_group(ev_subset, ep_subset, len(ev_subset)))
        overall_horizon_rows.append(row)
overall_horizon_summary_df = pd.DataFrame(overall_horizon_rows)

pd.set_option("display.width", 160)
display_cols = ["model", "patient_id", "horizon_minutes", "n_eligible_seizures", "seizure_sensitivity",
                 "false_alarms_per_hour", "time_in_warning_sec_mean", "auc_pr", "f1_preictal", "accuracy"]
print("=== PER PATIENT + HORIZON (primary + secondary metrics) ===")
print(patient_horizon_summary_df[display_cols].to_string(index=False))

print("\n=== OVERALL BY HORIZON (pooled across chb12 + chb24) ===")
overall_display_cols = ["model", "horizon_minutes", "n_eligible_seizures", "seizure_sensitivity",
                         "false_alarms_per_hour", "time_in_warning_sec_mean", "auc_pr", "f1_preictal", "accuracy"]
print(overall_horizon_summary_df[overall_display_cols].to_string(index=False))


=== PER PATIENT + HORIZON (primary + secondary metrics) ===
              model patient_id  horizon_minutes  n_eligible_seizures  seizure_sensitivity  false_alarms_per_hour  time_in_warning_sec_mean   auc_pr  f1_preictal  accuracy
logistic_regression      chb12                5                    5             1.000000                    NaN                272.200000      NaN     0.967742  0.937500
logistic_regression      chb12               15                    5             1.000000                    NaN                710.200000      NaN     0.974576  0.950413
logistic_regression      chb12               30                    5             1.000000                    NaN               1088.200000      NaN     0.972067  0.945652
logistic_regression      chb24                5                   14             0.928571              13.652695                200.153846 0.298502     0.346341  0.412281
logistic_regression      chb24               15                   14             0.92

## 10. Sanity checks: class distribution, majority-class behaviour, confusion matrices

Directly checking for the earlier project's known failure mode
(project_state.md Section 18.1: 78.9% accuracy while detecting almost
nothing) -- if Logistic Regression predicts essentially only one class on
TEST, that is flagged explicitly here, not hidden behind a high accuracy
number.


In [10]:

print("=== TEST-set predicted-positive rate (fraction of TEST epochs predicted PREICTAL) ===")
pos_rate_rows = []
for model_name in ["logistic_regression", "dummy_most_frequent"]:
    for H in HORIZONS:
        sub = predictions_df[(predictions_df["model"] == model_name) & (predictions_df["horizon_minutes"] == H)]
        pos_rate = sub["y_pred"].mean()
        true_pos_rate = sub["y_true"].mean()
        pos_rate_rows.append({"model": model_name, "horizon_minutes": H,
                               "predicted_positive_rate": pos_rate, "true_positive_rate": true_pos_rate})
pos_rate_df = pd.DataFrame(pos_rate_rows)
print(pos_rate_df.to_string(index=False))

flagged = pos_rate_df[
    (pos_rate_df["model"] == "logistic_regression")
    & ((pos_rate_df["predicted_positive_rate"] < 0.02) | (pos_rate_df["predicted_positive_rate"] > 0.98))
]
if len(flagged):
    print("\n[FLAG] Logistic Regression predicts essentially only one class at these horizons "
          "(likely repeating the old majority-class problem):")
    print(flagged.to_string(index=False))
else:
    print("\n[OK] Logistic Regression's predicted-positive rate is not degenerate (not <2% or >98%) "
          "at any horizon -- it is not trivially predicting only one class.")

print("\n=== Confusion matrices (pooled TEST epochs, per patient + horizon, Logistic Regression) ===")
confusion_rows = []
for pid in PATIENTS:
    for H in HORIZONS:
        sub = predictions_df[
            (predictions_df["model"] == "logistic_regression")
            & (predictions_df["patient_id"] == pid) & (predictions_df["horizon_minutes"] == H)
        ]
        cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        confusion_rows.append({"patient_id": pid, "horizon_minutes": H,
                                "TN": tn, "FP": fp, "FN": fn, "TP": tp})
        print(f"{pid} H={H:2d}min:  TN={tn:4d}  FP={fp:4d}  FN={fn:4d}  TP={tp:4d}")
confusion_df = pd.DataFrame(confusion_rows)

print("\n=== PREICTAL probability distribution (Logistic Regression), by TRUE label ===")
for H in HORIZONS:
    sub = predictions_df[(predictions_df["model"] == "logistic_regression") & (predictions_df["horizon_minutes"] == H)]
    print(f"\nH={H} min:")
    print(sub.groupby("y_true")["y_prob"].describe()[["count", "mean", "std", "min", "50%", "max"]].to_string())


=== TEST-set predicted-positive rate (fraction of TEST epochs predicted PREICTAL) ===
              model  horizon_minutes  predicted_positive_rate  true_positive_rate
logistic_regression                5                 0.660714            0.337302
logistic_regression               15                 0.794608            0.571245
logistic_regression               30                 0.788931            0.686679
dummy_most_frequent                5                 0.095238            0.337302
dummy_most_frequent               15                 0.667522            0.571245
dummy_most_frequent               30                 1.000000            0.686679

[OK] Logistic Regression's predicted-positive rate is not degenerate (not <2% or >98%) at any horizon -- it is not trivially predicting only one class.

=== Confusion matrices (pooled TEST epochs, per patient + horizon, Logistic Regression) ===
chb12 H= 5min:  TN=   0  FP=   0  FN=   3  TP=  45
chb12 H=15min:  TN=   0  FP=   0  FN=   6  

## 11. Naive-baseline comparison: Logistic Regression vs. always-majority-class

`DummyClassifier(strategy="most_frequent")`, fit on the exact same
per-fold TRAIN labels, predicts whichever class was more frequent in that
fold's TRAIN set (fit fresh per fold -- not always the same class, since
TRAIN prevalence varies by fold, Section 4). By construction it can only
ever get `seizure_sensitivity` > 0 in folds where PREICTAL happened to be
the TRAIN-majority class; comparing the two side by side is the direct
test for whether Logistic Regression is doing better than exploiting class
imbalance.


In [11]:

compare_cols = ["horizon_minutes", "seizure_sensitivity", "false_alarms_per_hour",
                "time_in_warning_sec_mean", "auc_pr", "accuracy", "f1_preictal"]
lr_overall = overall_horizon_summary_df[overall_horizon_summary_df["model"] == "logistic_regression"][compare_cols].set_index("horizon_minutes")
dummy_overall = overall_horizon_summary_df[overall_horizon_summary_df["model"] == "dummy_most_frequent"][compare_cols].set_index("horizon_minutes")
comparison_df = lr_overall.join(dummy_overall, lsuffix="_logreg", rsuffix="_dummy")
print("=== Logistic Regression vs. naive majority-class baseline (pooled overall, by horizon) ===")
print(comparison_df.to_string())


=== Logistic Regression vs. naive majority-class baseline (pooled overall, by horizon) ===
                 seizure_sensitivity_logreg  false_alarms_per_hour_logreg  time_in_warning_sec_mean_logreg  auc_pr_logreg  accuracy_logreg  f1_preictal_logreg  seizure_sensitivity_dummy  false_alarms_per_hour_dummy  time_in_warning_sec_mean_dummy  auc_pr_dummy  accuracy_dummy  f1_preictal_dummy
horizon_minutes                                                                                                                                                                                                                                                                                        
5                                  0.947368                     13.652695                       220.166667       0.535829         0.462302            0.461233                   0.263158                     0.000000                      272.200000      0.524416        0.757937           0.440367
15                   

## 12. Coefficient interpretation

Coefficients come from Logistic Regression fit on **standardized**
(per-fold, per-patient scaled) features, so magnitude is comparable within
any one fold's model. Comparing magnitude *across* folds/patients is
informative but each fold has its own independently-fit scaler and model --
summarized here as mean/std/sign-consistency across the 57 fold-level
models, not as one single "the" coefficient.

**These are NOT causal biomarkers.** A large logistic-regression
coefficient on standardized qEEG features reflects association with the
PREICTAL/INTERICTAL label in this specific labeling scheme and this
specific (small, 2-patient) sample -- not a validated neurophysiological
seizure precursor.


In [12]:

coef_summary_overall = (
    coefficients_df.groupby("feature")["coefficient"]
    .agg(mean_coefficient="mean", std_coefficient="std",
         mean_abs_coefficient=lambda s: s.abs().mean(),
         pct_folds_positive=lambda s: (s > 0).mean())
    .sort_values("mean_abs_coefficient", ascending=False)
    .reset_index()
)
print("=== Coefficient summary, pooled across ALL 57 folds (both patients, all horizons) ===")
print(coef_summary_overall.to_string(index=False))

coef_summary_by_horizon = (
    coefficients_df.groupby(["horizon_minutes", "feature"])["coefficient"]
    .agg(mean_coefficient="mean", std_coefficient="std",
         mean_abs_coefficient=lambda s: s.abs().mean(),
         pct_folds_positive=lambda s: (s > 0).mean())
    .reset_index()
)
coef_summary_by_patient_horizon = (
    coefficients_df.groupby(["patient_id", "horizon_minutes", "feature"])["coefficient"]
    .agg(mean_coefficient="mean", std_coefficient="std",
         mean_abs_coefficient=lambda s: s.abs().mean())
    .reset_index()
)

top_pos = coef_summary_overall.sort_values("mean_coefficient", ascending=False).head(5)
top_neg = coef_summary_overall.sort_values("mean_coefficient", ascending=True).head(5)
print("\nTop 5 features by most POSITIVE mean coefficient (higher -> more PREICTAL-associated):")
print(top_pos[["feature", "mean_coefficient", "std_coefficient", "pct_folds_positive"]].to_string(index=False))
print("\nTop 5 features by most NEGATIVE mean coefficient (higher -> more INTERICTAL-associated):")
print(top_neg[["feature", "mean_coefficient", "std_coefficient", "pct_folds_positive"]].to_string(index=False))

low_consistency = coef_summary_overall[
    (coef_summary_overall["pct_folds_positive"] > 0.3) & (coef_summary_overall["pct_folds_positive"] < 0.7)
]
print(f"\n[NOTE] {len(low_consistency)}/{len(coef_summary_overall)} features have inconsistent sign across folds "
      f"(positive in 30-70% of folds) -- their coefficient direction is not stable across held-out seizures/patients:")
print(low_consistency[["feature", "pct_folds_positive"]].to_string(index=False))


=== Coefficient summary, pooled across ALL 57 folds (both patients, all horizons) ===
          feature  mean_coefficient  std_coefficient  mean_abs_coefficient  pct_folds_positive
 delta_beta_ratio         -1.115606         1.143953              1.428622            0.298246
            alpha          1.230721         0.756289              1.284722            0.894737
         rel_beta         -0.351371         0.845259              0.704179            0.456140
            delta          0.615600         0.544251              0.700381            0.824561
             beta         -0.615859         0.370833              0.633554            0.052632
          entropy         -0.520767         0.506370              0.563304            0.105263
        rel_theta         -0.094839         0.634974              0.549677            0.350877
              var         -0.462678         0.488744              0.526056            0.228070
theta_alpha_ratio          0.437618         0.389948       

## 13. Diagnostic plots


In [13]:

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- PR curves, pooled overall per horizon ---
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4), sharey=True)
for ax, H in zip(axes, HORIZONS):
    for model_name, style in [("logistic_regression", "-"), ("dummy_most_frequent", "--")]:
        sub = predictions_df[(predictions_df["model"] == model_name) & (predictions_df["horizon_minutes"] == H)]
        if sub["y_true"].nunique() < 2:
            continue
        precision, recall, _ = precision_recall_curve(sub["y_true"], sub["y_prob"])
        ap = average_precision_score(sub["y_true"], sub["y_prob"])
        ax.plot(recall, precision, style, label=f"{model_name} (AP={ap:.3f})")
    ax.set_title(f"H = {H} min (pooled, both patients)")
    ax.set_xlabel("Recall")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Precision")
fig.suptitle("Precision-Recall curves -- pooled TEST epochs (overall, per horizon)")
fig.tight_layout()
fig.savefig(plot_dir / "pr_curves_overall_by_horizon.png", dpi=150)
plt.close(fig)

# --- Confusion matrix heatmaps, per patient + horizon (Logistic Regression) ---
fig, axes = plt.subplots(len(PATIENTS), len(HORIZONS), figsize=(4 * len(HORIZONS), 4 * len(PATIENTS)))
for i, pid in enumerate(PATIENTS):
    for j, H in enumerate(HORIZONS):
        ax = axes[i, j]
        sub = predictions_df[
            (predictions_df["model"] == "logistic_regression")
            & (predictions_df["patient_id"] == pid) & (predictions_df["horizon_minutes"] == H)
        ]
        cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
        im = ax.imshow(cm, cmap="Blues")
        for r in range(2):
            for c in range(2):
                ax.text(c, r, str(cm[r, c]), ha="center", va="center")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["INTERICTAL", "PREICTAL"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["INTERICTAL", "PREICTAL"])
        ax.set_title(f"{pid}, H={H}min")
        if j == 0:
            ax.set_ylabel("True")
        if i == len(PATIENTS) - 1:
            ax.set_xlabel("Predicted")
fig.suptitle("Confusion matrices (pooled TEST epochs, Logistic Regression)")
fig.tight_layout()
fig.savefig(plot_dir / "confusion_matrices_logreg.png", dpi=150)
plt.close(fig)

# --- PREICTAL probability distributions, per horizon, split by true label ---
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4), sharey=True)
for ax, H in zip(axes, HORIZONS):
    sub = predictions_df[(predictions_df["model"] == "logistic_regression") & (predictions_df["horizon_minutes"] == H)]
    ax.hist(sub.loc[sub["y_true"] == 0, "y_prob"], bins=20, alpha=0.6, label="true INTERICTAL", density=True)
    ax.hist(sub.loc[sub["y_true"] == 1, "y_prob"], bins=20, alpha=0.6, label="true PREICTAL", density=True)
    ax.axvline(THRESHOLD, color="black", linestyle=":", label=f"threshold={THRESHOLD}")
    ax.set_title(f"H = {H} min")
    ax.set_xlabel("Predicted P(PREICTAL)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Density")
fig.suptitle("Predicted PREICTAL probability by true label (Logistic Regression, pooled)")
fig.tight_layout()
fig.savefig(plot_dir / "probability_distributions.png", dpi=150)
plt.close(fig)

# --- Primary-metric comparison across horizons, per patient (LR vs dummy) ---
metric_names = ["seizure_sensitivity", "false_alarms_per_hour", "auc_pr"]
fig, axes = plt.subplots(len(PATIENTS), len(metric_names), figsize=(4.5 * len(metric_names), 4 * len(PATIENTS)))
for i, pid in enumerate(PATIENTS):
    for j, metric in enumerate(metric_names):
        ax = axes[i, j]
        for model_name, marker in [("logistic_regression", "o-"), ("dummy_most_frequent", "s--")]:
            sub = patient_horizon_summary_df[
                (patient_horizon_summary_df["model"] == model_name) & (patient_horizon_summary_df["patient_id"] == pid)
            ].sort_values("horizon_minutes")
            ax.plot(sub["horizon_minutes"], sub[metric], marker, label=model_name)
        ax.set_title(f"{pid}: {metric}")
        ax.set_xlabel("Horizon (min)")
        ax.set_xticks(HORIZONS)
        if j == 0:
            ax.legend(fontsize=7)
fig.suptitle("Primary metrics across horizons, per patient")
fig.tight_layout()
fig.savefig(plot_dir / "primary_metrics_by_horizon.png", dpi=150)
plt.close(fig)

print("[OK] Saved 4 diagnostic plot files to", plot_dir)
for p in sorted(plot_dir.glob("*.png")):
    print(" -", p.name)


[OK] Saved 4 diagnostic plot files to C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\modeling\logistic_loso_binary\plots
 - confusion_matrices_logreg.png
 - pr_curves_overall_by_horizon.png
 - primary_metrics_by_horizon.png
 - probability_distributions.png


## 14. Save outputs

New, clearly-named directory: `outputs/modeling/logistic_loso_binary/`.
Nothing existing is overwritten.


In [14]:

predictions_df.to_parquet(run_dir / "predictions.parquet", index=False)
predictions_df.to_csv(run_dir / "predictions.csv", index=False)

fold_metrics_df.to_csv(run_dir / "fold_metrics.csv", index=False)
patient_horizon_summary_df.to_csv(run_dir / "patient_horizon_summary.csv", index=False)
overall_horizon_summary_df.to_csv(run_dir / "overall_horizon_summary.csv", index=False)

coefficients_df.to_csv(run_dir / "coefficients_long.csv", index=False)
coef_summary_overall.to_csv(run_dir / "coefficients_summary_overall.csv", index=False)
coef_summary_by_horizon.to_csv(run_dir / "coefficients_summary_by_horizon.csv", index=False)
coef_summary_by_patient_horizon.to_csv(run_dir / "coefficients_summary_by_patient_horizon.csv", index=False)

import json as _json
experiment_config = {
    "task": "Binary PREICTAL vs INTERICTAL detection, patient-specific LOSO, Logistic Regression baseline",
    "model": "LogisticRegression",
    "model_params": LOGREG_PARAMS,
    "naive_baseline": "DummyClassifier(strategy='most_frequent')",
    "random_state": RANDOM_STATE,
    "decision_threshold": THRESHOLD,
    "threshold_tuned_on_test": False,
    "horizons_minutes": HORIZONS,
    "patients": PATIENTS,
    "patients_pooled_across": False,
    "feature_cols": FEATURE_COLS,
    "epoch_sec": EPOCH_SEC,
    "n_folds_total": int(n_folds_built),
    "source_labels_table": str(in_path),
    "source_loso_manifest_crosscheck": str(manifest_path),
    "normalization": "per-fold, per-patient StandardScaler fit on TRAIN rows only (reused verbatim from EEG_patient_normalization.ipynb / EEG_loso_evaluation_splitting.ipynb)",
    "metrics_framework_source": "EEG_seizure_prediction_metrics.ipynb",
}
with open(run_dir / "experiment_config.json", "w") as f:
    _json.dump(experiment_config, f, indent=2, default=str)

print("Saved to", run_dir, ":")
for p in sorted(run_dir.glob("*")):
    if p.is_file():
        print(" -", p.name)

import os as _os
for f in ["labels_binary_preictal.parquet", "features_patient_scaled.parquet",
          "loso_fold_manifest.parquet", "features_chb24_forecasting.parquet",
          "features_chb12_forecasting.parquet"]:
    assert _os.path.exists(output_dir / f)
print("\n[OK] All previously existing outputs remain untouched.")


Saved to C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\modeling\logistic_loso_binary :
 - coefficients_long.csv
 - coefficients_summary_by_horizon.csv
 - coefficients_summary_by_patient_horizon.csv
 - coefficients_summary_overall.csv
 - experiment_config.json
 - fold_metrics.csv
 - overall_horizon_summary.csv
 - patient_horizon_summary.csv
 - predictions.csv
 - predictions.parquet

[OK] All previously existing outputs remain untouched.


## 15. Conclusion


In [15]:

from IPython.display import display, Markdown

lr_5 = overall_horizon_summary_df[(overall_horizon_summary_df.model=="logistic_regression") & (overall_horizon_summary_df.horizon_minutes==5)].iloc[0]
lr_15 = overall_horizon_summary_df[(overall_horizon_summary_df.model=="logistic_regression") & (overall_horizon_summary_df.horizon_minutes==15)].iloc[0]
lr_30 = overall_horizon_summary_df[(overall_horizon_summary_df.model=="logistic_regression") & (overall_horizon_summary_df.horizon_minutes==30)].iloc[0]
dummy_5 = overall_horizon_summary_df[(overall_horizon_summary_df.model=="dummy_most_frequent") & (overall_horizon_summary_df.horizon_minutes==5)].iloc[0]

conclusion = f"""
### A. Did Logistic Regression detect seizures, or exploit class imbalance?

Pooled (overall, both patients) results:

| Horizon | Seizure sensitivity (LR) | FA/hour (LR) | AUC-PR (LR) | Sensitivity (dummy) |
|---|---|---|---|---|
| 5 min  | {lr_5['seizure_sensitivity']:.3f} ({lr_5['n_seizures_detected']}/{lr_5['n_folds']}) | {lr_5['false_alarms_per_hour']:.3f} | {lr_5['auc_pr']:.3f} | {dummy_5['seizure_sensitivity']:.3f} |
| 15 min | {lr_15['seizure_sensitivity']:.3f} ({lr_15['n_seizures_detected']}/{lr_15['n_folds']}) | {lr_15['false_alarms_per_hour']:.3f} | {lr_15['auc_pr']:.3f} | -- |
| 30 min | {lr_30['seizure_sensitivity']:.3f} ({lr_30['n_seizures_detected']}/{lr_30['n_folds']}) | {lr_30['false_alarms_per_hour']:.3f} | {lr_30['auc_pr']:.3f} | -- |

Predicted-positive rate check (Section 10) found: {"no degenerate single-class prediction" if len(flagged)==0 else f"{len(flagged)} horizon(s) with near-degenerate prediction -- see flagged table above"}.

### B. What we can and cannot conclude from this first baseline

This is a **first, minimally-tuned baseline** on **2 patients** (chb12,
chb24), **19 total eligible held-out seizures** across all three horizons
combined, several of which contribute a TEST set with 0 interictal epochs
(project_state.md Section 20.5) -- so per-fold and even some per-patient
metrics above are individually noisy, and false-alarm/AUC-PR figures for
CHB12 specifically are undefined at the pooled level for that reason (see
the patient+horizon table). Coefficient signs are not fully consistent
across folds (Section 12) -- feature importance here is descriptive of
*this* baseline on *this* sample, not a validated biomarker claim.

### C. Next step

Random Forest and TCN experiments on these same LOSO folds are the natural
next comparisons (both explicitly out of scope for this task). A
channel-wise or larger-cohort expansion remains future work.
"""
display(Markdown(conclusion))



### A. Did Logistic Regression detect seizures, or exploit class imbalance?

Pooled (overall, both patients) results:

| Horizon | Seizure sensitivity (LR) | FA/hour (LR) | AUC-PR (LR) | Sensitivity (dummy) |
|---|---|---|---|---|
| 5 min  | 0.947 (18/19) | 13.653 | 0.536 | 0.263 |
| 15 min | 0.947 (18/19) | 13.293 | 0.650 | -- |
| 30 min | 0.947 (18/19) | 12.575 | 0.736 | -- |

Predicted-positive rate check (Section 10) found: no degenerate single-class prediction.

### B. What we can and cannot conclude from this first baseline

This is a **first, minimally-tuned baseline** on **2 patients** (chb12,
chb24), **19 total eligible held-out seizures** across all three horizons
combined, several of which contribute a TEST set with 0 interictal epochs
(project_state.md Section 20.5) -- so per-fold and even some per-patient
metrics above are individually noisy, and false-alarm/AUC-PR figures for
CHB12 specifically are undefined at the pooled level for that reason (see
the patient+horizon table). Coefficient signs are not fully consistent
across folds (Section 12) -- feature importance here is descriptive of
*this* baseline on *this* sample, not a validated biomarker claim.

### C. Next step

Random Forest and TCN experiments on these same LOSO folds are the natural
next comparisons (both explicitly out of scope for this task). A
channel-wise or larger-cohort expansion remains future work.
